# 01 — Clean and Merge

Build the three final dataframes (`df_pistol`, `df_post_pistol`, `df_normal`) from the raw per-map CSVs produced by `parsing/parser.py` and join them with HLTV team statistics scraped via `scraping/`.

**Pipeline overview**

- **I. Cleaning raw demo data** — concatenate per-map CSVs, fix phantom rounds, recalculate scores/streaks, drop 4v5 pistols, save the cleaned `df_rounds_raw.csv`.
- **II. Splitting by round type** — normalize team names, split into pistol / post-pistol / normal, drop the columns that carry no signal per regime.
- **III. Joining HLTV statistics** — merge the team statistics scraped from HLTV.org and fill the resulting NaN with a three-step fallback chain.
- **IV. Saving outputs** — write the three final dataframes to `data/processed/`.


## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd

BASE_DIR   = Path("..")
PARSED_DIR = BASE_DIR / "data" / "parsed"
STATS_DIR  = BASE_DIR / "data" / "stats"
RAW_PATH   = BASE_DIR / "data" / "df_rounds_raw.csv"
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORRUPTED_DEMOS = {
    "mouz-vs-liquid-inferno",
    "mouz-vs-furia-mirage",
    "parivision-vs-astralis-dust2",
    "mouz-vs-parivision-dust2",
}


## I. Cleaning raw demo data


### I.1 Concatenate parsed CSVs (or load cached `df_rounds_raw.csv`)

Iterate over every per-map CSV produced by the parser. Four demos that are structurally corrupted on HLTV are skipped.

**Fast path**: if `data/df_rounds_raw.csv` already exists from a previous run, the cell below loads it directly and Steps 2 to 5 are skipped (their code is guarded by a `SKIP_CLEANING` flag).

In [2]:
if RAW_PATH.exists():
    print(f"Found existing {RAW_PATH} - loading it and skipping Steps 1 to 5.")
    df = pd.read_csv(RAW_PATH)
    SKIP_CLEANING = True
else:
    SKIP_CLEANING = False
    csvs = sorted(PARSED_DIR.rglob("*.csv"))
    print(f"Found {len(csvs)} parsed CSV files")
    if not csvs:
        raise FileNotFoundError(
            f"No CSVs in {PARSED_DIR}. Run the parser first to produce data/parsed/"
        )

    dfs = []
    for f in csvs:
        if any(c in f.stem for c in CORRUPTED_DEMOS):
            print(f"  SKIP corrupted: {f.stem}")
            continue
        rel = f.relative_to(PARSED_DIR)
        parts = rel.parts
        tmp = pd.read_csv(f)
        if "tournament" not in tmp.columns:
            tmp["tournament"] = parts[0] if len(parts) >= 1 else "unknown"
        if "match_date" not in tmp.columns:
            tmp["match_date"] = parts[1] if len(parts) >= 2 else "unknown"
        if "match_file" not in tmp.columns:
            tmp["match_file"] = f.stem
        dfs.append(tmp)

    df = pd.concat(dfs, ignore_index=True)
    print(f"Concatenated {len(df)} rows from {len(dfs)} files")


Found existing ..\data\df_rounds_raw.csv - loading it and skipping Steps 1 to 5.


### I.2 Align rounds on pistol anchors

The parser sometimes emits phantom rounds at the start of a match (warmup, knife round, forfeit, mid-game start). These shift every subsequent `round_number`.

For each match we locate the real R1 with a *pistol shape* test: both teams' `money_total` falls within `[4000, 6500]`. The lower bound tolerates the late-snapshot drift of the `fissure-playground-2` tournament. Matches with no pistol anchor in the first 15 rows are dropped (corrupted / forfeit). Same procedure for R13 (expected position 12).

In [4]:
if not SKIP_CLEANING:
    R1_SEARCH_WINDOW   = 15
    R13_EXPECTED_POS   = 12
    R13_SEARCH_WINDOW  = 6
    PISTOL_SHAPE_LO    = 4000
    PISTOL_SHAPE_HI    = 6500


    def is_pistol_shape(row):
        return (PISTOL_SHAPE_LO <= row["ct_money_total"] <= PISTOL_SHAPE_HI
                and PISTOL_SHAPE_LO <= row["t_money_total"] <= PISTOL_SHAPE_HI)


    def find_exact_pistol(df_m, start_idx, max_window):
        end = min(start_idx + max_window, len(df_m))
        for i in range(start_idx, end):
            row = df_m.iloc[i]
            if row["ct_money_total"] == 5000.0 and row["t_money_total"] == 5000.0:
                return i
        return None


    def fix_match(df_m):
        df_m = df_m.sort_values("round_number").reset_index(drop=True)

        if is_pistol_shape(df_m.iloc[0]):
            r1_idx = 0
            status = "normal"
        else:
            r1_idx = find_exact_pistol(df_m, 1, R1_SEARCH_WINDOW - 1)
            if r1_idx is None:
                return None, "broken_no_r1"
            status = f"phantom_r1_dropped={r1_idx}"

        df_m = df_m.iloc[r1_idx:].reset_index(drop=True)

        if len(df_m) > R13_EXPECTED_POS:
            if not is_pistol_shape(df_m.iloc[R13_EXPECTED_POS]):
                r13_idx = find_exact_pistol(df_m, R13_EXPECTED_POS + 1, R13_SEARCH_WINDOW - 1)
                if r13_idx is None:
                    return None, f"broken_no_r13 (after {status})"
                n_drop = r13_idx - R13_EXPECTED_POS
                df_m = pd.concat(
                    [df_m.iloc[:R13_EXPECTED_POS], df_m.iloc[r13_idx:]],
                    ignore_index=True,
                )
                status += f"+phantom_r13_dropped={n_drop}"

        df_m = df_m.copy()
        df_m["round_number"] = range(1, len(df_m) + 1)
        return df_m, status


    fixed_dfs = []
    status_counts = {}
    for _, group in df.groupby(["tournament", "match_date", "match_file"]):
        fixed, status = fix_match(group)
        status_counts[status] = status_counts.get(status, 0) + 1
        if fixed is not None:
            fixed_dfs.append(fixed)

    df = pd.concat(fixed_dfs, ignore_index=True)

    print("Alignment status counts:")
    for k, v in sorted(status_counts.items()):
        print(f"  {k}: {v}")
    print(f"\nTotal rows after alignment: {len(df)}")

else:
    pass


### I.3 Recalculate scores, streaks, overtime

After dropping phantom rounds, the parser's running scores and streaks no longer match the cleaned sequence. We recompute everything from the `round_winner` column.

In [5]:
if not SKIP_CLEANING:
    df = df.sort_values(["tournament", "match_date", "match_file", "round_number"]).reset_index(drop=True)

    new_ct_score, new_t_score = [], []
    new_ct_ws, new_ct_ls, new_t_ws, new_t_ls = [], [], [], []
    new_is_overtime = []

    prev_match = None
    ct_s = t_s = ct_ws = ct_ls = t_ws = t_ls = 0

    for _, row in df.iterrows():
        match_key = (row["tournament"], row["match_date"], row["match_file"])
        rn = row["round_number"]

        if match_key != prev_match:
            ct_s = t_s = ct_ws = ct_ls = t_ws = t_ls = 0
            prev_match = match_key

        if rn in (13, 25, 28, 31, 34, 37, 40):
            ct_ws = ct_ls = t_ws = t_ls = 0

        new_ct_score.append(ct_s)
        new_t_score.append(t_s)
        new_ct_ws.append(ct_ws)
        new_ct_ls.append(ct_ls)
        new_t_ws.append(t_ws)
        new_t_ls.append(t_ls)
        new_is_overtime.append(1 if rn > 24 else 0)

        if row["round_winner"] == 1:
            ct_s += 1; ct_ws += 1; ct_ls = 0; t_ws = 0; t_ls += 1
        else:
            t_s += 1; t_ws += 1; t_ls = 0; ct_ws = 0; ct_ls += 1

    df["ct_score"] = new_ct_score
    df["t_score"]  = new_t_score
    df["ct_rounds_won_streak"]  = new_ct_ws
    df["ct_rounds_lost_streak"] = new_ct_ls
    df["t_rounds_won_streak"]   = new_t_ws
    df["t_rounds_lost_streak"]  = new_t_ls
    df["is_overtime"]           = new_is_overtime

    print("Scores, streaks and overtime flag recomputed.")

else:
    pass


### I.4 Drop invalid pistol rows (4v5)

A pistol round with exactly `$4000` on either side corresponds to a 4 vs 5 case (one player missing: `4 x $800 cash + 4 x $200 pistol = $4000`). These are not real pistol rounds and are removed. The slight-drift cases from `fissure-playground-2` (e.g. 4800-4800) are kept since all 10 players are present.

In [6]:
if not SKIP_CLEANING:
    bad_pistol_mask = (
        df["round_number"].isin([1, 13])
        & ((df["ct_money_total"] == 4000) | (df["t_money_total"] == 4000))
    )
    print(f"Dropping {bad_pistol_mask.sum()} invalid pistol rows (4v5)")
    df = df[~bad_pistol_mask].reset_index(drop=True)

    r1r13 = df[df["round_number"].isin([1, 13])]
    out_of_range = r1r13[
        (r1r13["ct_money_total"] < PISTOL_SHAPE_LO) | (r1r13["ct_money_total"] > PISTOL_SHAPE_HI)
        | (r1r13["t_money_total"]  < PISTOL_SHAPE_LO) | (r1r13["t_money_total"]  > PISTOL_SHAPE_HI)
    ]
    assert len(out_of_range) == 0, f"{len(out_of_range)} R1/R13 rows outside pistol range"
    print(f"Pistol integrity OK: {(df['round_number'] == 1).sum()} R1s, {(df['round_number'] == 13).sum()} R13s")

else:
    pass


### I.5 Save the intermediate `df_rounds_raw.csv`

In [7]:
if not SKIP_CLEANING:
    df.to_csv(RAW_PATH, index=False)
    print(f"Saved {RAW_PATH} ({df.shape})")

else:
    pass


## II. Splitting by round type


### II.1 Normalize team names

Team names extracted from demos do not always match the names used on HLTV (sponsor changes, abbreviations, casing). A hand-curated mapping is applied before any join.

In [8]:
TEAM_NAME_MAP = {
    "Aurora Gaming": "Aurora",
    "B8 Esports": "B8",
    "BC Game Esports": "BC.Game",
    "BCG": "BC.Game",
    "Betclic Apogee": "Betclic",
    "FUT Esports": "FUT",
    "FaZe Clan": "FaZe",
    "FaZe Clan CS2": "FaZe",
    "Fnatic": "fnatic",
    "G2 Esports": "G2",
    "G2 Esports CS2": "G2",
    "Gamdom Imperial": "Imperial",
    "HOTU Esports": "HOTU",
    "Imperial Esports": "Imperial",
    "LEGACY": "Legacy",
    "Legacy_": "Legacy",
    "Lynn Vision Gaming": "Lynn Vision",
    "M80 CS2": "M80",
    "MongolZ": "The MongolZ",
    "Natus Vincere NAVI": "Natus Vincere",
    "RARE ATOM": "Rare Atom",
    "Team Falcons": "Falcons",
    "Team Liquid": "Liquid",
    "Team Spirit": "Spirit",
    "Team Vitality": "Vitality",
    "Team Vitality CS2": "Vitality",
    "Tyloo": "TYLOO",
    "V.P.": "Virtus.pro",
    "Virtus.Pro": "Virtus.pro",
    "paiN Gaming": "paiN",
}

df["ct_team_name"] = df["ct_team_name"].replace(TEAM_NAME_MAP)
df["t_team_name"]  = df["t_team_name"].replace(TEAM_NAME_MAP)
print("Team names normalized.")


Team names normalized.


### II.2 Split by round type

The economic and tactical dynamics differ enough between pistol, post-pistol and normal rounds to justify a separate dataframe for each.

In [9]:
df["round_type"] = df["round_number"].apply(
    lambda r: "pistol" if r in (1, 13)
    else "post_pistol" if r in (2, 14)
    else "normal"
)
df_pistol      = df[df["round_type"] == "pistol"].copy()
df_post_pistol = df[df["round_type"] == "post_pistol"].copy()
df_normal      = df[df["round_type"] == "normal"].copy()

print(f"pistol      : {len(df_pistol)}")
print(f"post_pistol : {len(df_post_pistol)}")
print(f"normal      : {len(df_normal)}")


pistol      : 2945
post_pistol : 2956
normal      : 26322


### II.3 Drop irrelevant demo columns per split

Several columns produced by the parser carry no information for a given regime (e.g. streaks on pistol rounds since the half resets the state) or are direct duplicates (e.g. `lost_streak` mirrors the opposite side's `won_streak`).

In [10]:
DROP_PISTOL = [
    "ct_rounds_won_streak", "ct_rounds_lost_streak",
    "t_rounds_won_streak",  "t_rounds_lost_streak",
    "is_overtime",
    "ct_cash", "t_cash", "ct_money_total", "t_money_total",
    "ct_awp_count", "t_awp_count",
    "ct_ssg_count", "t_ssg_count",
    "ct_rifle_count", "t_rifle_count",
    "ct_smg_count", "t_smg_count",
    "ct_heavy_count", "t_heavy_count",
    "ct_ak_count",
    "ct_survivors_previous", "t_survivors_previous",
    "ct_equipment_saved_value", "t_equipment_saved_value",
]
DROP_POST_PISTOL = ["ct_ak_count", "ct_rounds_lost_streak", "t_rounds_lost_streak"]
DROP_NORMAL      = ["ct_rounds_lost_streak", "t_rounds_lost_streak"]

df_pistol      = df_pistol.drop(columns=[c for c in DROP_PISTOL      if c in df_pistol.columns])
df_post_pistol = df_post_pistol.drop(columns=[c for c in DROP_POST_PISTOL if c in df_post_pistol.columns])
df_normal      = df_normal.drop(columns=[c for c in DROP_NORMAL      if c in df_normal.columns])

print(f"pistol cols      : {len(df_pistol.columns)}")
print(f"post_pistol cols : {len(df_post_pistol.columns)}")
print(f"normal cols      : {len(df_normal.columns)}")


pistol cols      : 34
post_pistol cols : 55
normal cols      : 56


## III. Joining HLTV statistics


### III.1 Merge HLTV team statistics

For each round we attach the historical statistics of both teams. The HLTV scrape produces, for every team, a 4-dimensional grid of stats indexed by:

- **window**: `30d`, `90d`, `6months`
- **scope**:  `global` (all maps) and `map` (the map currently played)
- **side**:   `CT`, `T`
- **top**:    opponent tier (`Top5`, `Top10`, `Top20`, `Top30`, `Top50`)

The correct `top` for a given match comes from `matches.json` (HLTV's own classification). The stats kept depend on the split: pistol-specific stats for pistol, round-2 conversion for post-pistol, generic ratings for normal.

In [11]:
_team_csv_cache     = {}
_matches_json_cache = {}


def _load_team_csv(tournament, window, match_date, team):
    key = (tournament, window, match_date, team)
    if key not in _team_csv_cache:
        path = STATS_DIR / tournament / "teams" / window / match_date / f"{team}.csv"
        _team_csv_cache[key] = pd.read_csv(path, keep_default_na=False) if path.exists() else None
    return _team_csv_cache[key]


def _load_matches_json(tournament, match_date):
    key = (tournament, match_date)
    if key not in _matches_json_cache:
        path = STATS_DIR / tournament / "matches" / match_date / "matches.json"
        _matches_json_cache[key] = json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    return _matches_json_cache[key]


def _get_top(tournament, match_date, map_name, window, ct_team, t_team):
    data = _load_matches_json(tournament, match_date)
    if data is None:
        return None
    for m in data["matches"]:
        teams = {m["team1"], m["team2"]}
        if ct_team in teams and t_team in teams and map_name in m.get("tops", {}):
            raw = m["tops"][map_name].get(window)
            if raw:
                return raw.lower()
    return None


def merge_hltv_stats(df_in, stats_to_merge, label):
    windows = ["30d", "90d", "6months"]
    match_keys = df_in[["tournament", "match_date", "map_name", "ct_team_name", "t_team_name"]].drop_duplicates()
    print(f"  {label}: {len(match_keys)} unique match-maps to look up")

    lookup = {}
    missing_top = missing_csv = 0
    for _, mk in match_keys.iterrows():
        tour, date, map_name, ct_team, t_team = (
            mk["tournament"], mk["match_date"], mk["map_name"], mk["ct_team_name"], mk["t_team_name"]
        )
        cols = {}
        for window in windows:
            top = _get_top(tour, date, map_name, window, ct_team, t_team)
            if top is None:
                missing_top += 1
                continue
            for side_label, prefix, team in [("CT", "ct", ct_team), ("T", "t", t_team)]:
                team_df = _load_team_csv(tour, window, date, team)
                if team_df is None:
                    missing_csv += 1
                    continue
                for scope, map_filter in [("global", "global"), ("map", map_name)]:
                    filt = team_df[
                        (team_df["map"] == map_filter)
                        & (team_df["side"] == side_label)
                        & (team_df["top"] == top)
                    ]
                    for stat in stats_to_merge:
                        col_name = f"{prefix}_team_{scope}_{window}_{stat}"
                        if len(filt) > 0 and stat in filt.columns:
                            val = filt[stat].values[0]
                            if isinstance(val, str) and val in ("N/A", ""):
                                val = None
                            else:
                                try:
                                    val = float(val)
                                except (ValueError, TypeError):
                                    val = None
                            cols[col_name] = val
                        else:
                            cols[col_name] = None
        lookup[(tour, date, map_name, ct_team, t_team)] = cols

    if missing_top:
        print(f"    {missing_top} match-map-window combos missing 'top' in matches.json")
    if missing_csv:
        print(f"    {missing_csv} team CSVs not found")

    new_rows = [
        lookup.get((r["tournament"], r["match_date"], r["map_name"], r["ct_team_name"], r["t_team_name"]), {})
        for _, r in df_in.iterrows()
    ]
    result = pd.DataFrame(new_rows, index=df_in.index)
    for col in result.columns:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    return pd.concat([df_in, result], axis=1)


print("Merging HLTV stats into each split...")
df_pistol      = merge_hltv_stats(df_pistol,      ["rating", "fa", "pistol_win_pct"],             "pistol")
df_post_pistol = merge_hltv_stats(df_post_pistol, ["rating", "fa", "round2_conv", "round2_break"], "post_pistol")
df_normal      = merge_hltv_stats(df_normal,      ["rating", "rw_pct", "adr", "fa"],              "normal")


Merging HLTV stats into each split...
  pistol: 1519 unique match-maps to look up
    33 match-map-window combos missing 'top' in matches.json
  post_pistol: 2956 unique match-maps to look up
    66 match-map-window combos missing 'top' in matches.json
  normal: 2942 unique match-maps to look up
    66 match-map-window combos missing 'top' in matches.json


### III.2 NaN fallback chain

HLTV does not publish every `(team, map, side, window, top)` combination. The remaining NaN are filled with a three-step cascade:

1. **map -> global**: replace a missing map-specific value by the all-maps value    of the same team, same side, same window.
2. **cross-window**: fall back to `90d` then `6months` if `30d` is missing.
3. **expanding median by date**: median over all past matches (no leakage).

The arbitrary nature of these choices is acknowledged in the report; alternative strategies (e.g. a single neutral value such as `1.00` for the rating) would also be defensible.

In [14]:
window_order = ["30d", "90d", "6months"]
splits = {"pistol": df_pistol, "post_pistol": df_post_pistol, "normal": df_normal}

for name in splits:
    split_df = splits[name]
    hltv_cols = [c for c in split_df.columns if "_team_map_" in c or "_team_global_" in c]
    nan_before = split_df[hltv_cols].isna().sum().sum()

    map_cols = [c for c in split_df.columns if "_team_map_" in c]
    filled_global = 0
    for mc in map_cols:
        gc = mc.replace("_team_map_", "_team_global_")
        if gc in split_df.columns:
            mask = split_df[mc].isna()
            filled_global += mask.sum()
            split_df[mc] = split_df[mc].fillna(split_df[gc])

    filled_cross = 0
    for scope in ["map", "global"]:
        for stat_col in hltv_cols:
            if f"_team_{scope}_" not in stat_col:
                continue
            col_window = next((w for w in window_order if f"_{w}_" in stat_col), None)
            if col_window is None:
                continue
            w_idx = window_order.index(col_window)
            for fb in window_order[w_idx + 1:]:
                fallback_col = stat_col.replace(f"_{col_window}_", f"_{fb}_")
                if fallback_col in split_df.columns:
                    mask = split_df[stat_col].isna()
                    filled_cross += mask.sum()
                    split_df[stat_col] = split_df[stat_col].fillna(split_df[fallback_col])

    hltv_cols = [c for c in split_df.columns if "_team_map_" in c or "_team_global_" in c]
    remaining_before = split_df[hltv_cols].isna().sum().sum()
    split_df = split_df.sort_values("match_date").reset_index(drop=True)
    expanding_med = split_df[hltv_cols].expanding(min_periods=1).median()
    for col in hltv_cols:
        mask = split_df[col].isna()
        split_df.loc[mask, col] = expanding_med.loc[mask, col]

    remaining_after = split_df[hltv_cols].isna().sum().sum()
    if remaining_after > 0:
        for col in hltv_cols:
            if split_df[col].isna().any():
                first_valid = split_df[col].first_valid_index()
                if first_valid is not None:
                    split_df[col] = split_df[col].fillna(split_df[col].iloc[first_valid])

    nan_after = split_df[hltv_cols].isna().sum().sum()
    print(f"\n{name}: {nan_before} initial NaN")
    print(f"  map -> global      : {filled_global} filled")
    print(f"  cross-window       : {filled_cross} filled")
    print(f"  expanding median   : {remaining_before - remaining_after} filled")
    print(f"  remaining NaN      : {nan_after}")
    splits[name] = split_df

df_pistol      = splits["pistol"]
df_post_pistol = splits["post_pistol"]
df_normal      = splits["normal"]



pistol: 0 initial NaN
  map -> global      : 0 filled
  cross-window       : 0 filled
  expanding median   : 0 filled
  remaining NaN      : 0

post_pistol: 0 initial NaN
  map -> global      : 0 filled
  cross-window       : 0 filled
  expanding median   : 0 filled
  remaining NaN      : 0

normal: 0 initial NaN
  map -> global      : 0 filled
  cross-window       : 0 filled
  expanding median   : 0 filled
  remaining NaN      : 0


## IV. Saving outputs


### IV.1 Save the three final dataframes

In [15]:
for name, d in [("pistol", df_pistol), ("post_pistol", df_post_pistol), ("normal", df_normal)]:
    out = OUTPUT_DIR / f"df_{name}.csv"
    d.to_csv(out, index=False)
    print(f"{name:12s} -> {out}  shape={d.shape}  CT winrate={d['round_winner'].mean():.3f}")

total = len(df_pistol) + len(df_post_pistol) + len(df_normal)
print(f"\nTotal: {total} rounds across 3 splits")


pistol       -> ..\data\processed\df_pistol.csv  shape=(2945, 70)  CT winrate=0.505
post_pistol  -> ..\data\processed\df_post_pistol.csv  shape=(2956, 103)  CT winrate=0.523
normal       -> ..\data\processed\df_normal.csv  shape=(26322, 104)  CT winrate=0.511

Total: 32223 rounds across 3 splits
